# Quant Insights: Second and Third-Order Market Observations

This notebook tests the deterministic scoring and ranking of stocks based on market microstructure, behavioral panic/exhaustion, and institutional supply digestion, rather than retail indicators.

## 1. Import Libraries and Load Data

In [1]:
import json
import pandas as pd
import numpy as np

# Load historical F&O dump
db_path = 'screener/data/fo_historical_dump.json'
with open(db_path, 'r', encoding='utf-8') as f:
    dump_data = json.load(f)

print(f"Loaded {len(dump_data)} symbols from historical dump.")

Loaded 115 symbols from historical dump.


## 2. Preprocess Data and Create DataFrames

We will convert the raw candle arrays into Pandas DataFrames. Each candle contains: `[date, open, high, low, close, volume]`.

In [9]:
dfs = {}
for symbol, candles in dump_data.items():
    if not candles:
        continue
    df = pd.DataFrame(candles, columns=['date', 'open', 'high', 'low', 'close', 'volume'])
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    # Convert columns to numeric
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df.sort_index(inplace=True)
    dfs[symbol] = df

print(f"Parsed {len(dfs)} active DataFrames.")

Parsed 115 active DataFrames.


## 3. Extract Market Benchmark (Nifty 50) Returns

We will use `'NIFTY 50'` as the benchmark for market returns and beta calculations.

In [10]:
nifty_df = dfs.get('NIFTY 50')
if nifty_df is not None:
    nifty_df['market_return'] = nifty_df['close'].pct_change()
    print("Nifty 50 benchmark returns computed successfully.")
    print(nifty_df[['close', 'market_return']].tail(3))
else:
    print("NIFTY 50 not found in database! Please check the keys.")

Nifty 50 benchmark returns computed successfully.
                              close  market_return
date                                              
2026-08-26 00:00:00+05:30  24207.75      -0.005211
2026-08-27 00:00:00+05:30  24090.85      -0.004829
2026-08-28 00:00:00+05:30  24175.65       0.003520


## 4. Calculate Rolling Indicators and Beta for Stocks

We calculate:
- Daily stock return ($R_s$)
- 60-day rolling beta ($\beta$) against Nifty 50
- Residual return ($e_s = R_s - \beta \cdot R_m$)
- Close Location Value ($CLV = \frac{(Close-Low) - (High-Close)}{High-Low}$)
- 20-day Volume SMA and standard deviation
- Volume Z-score
- Spread (High - Low) and its 20-day SMA and standard deviation
- Spread Z-score
- 10-day Close EMA to determine short-term trend

In [11]:
for symbol, df in dfs.items():
    if symbol == 'NIFTY 50' or 'NIFTY' in symbol:
        continue
    
    # Daily return
    df['return'] = df['close'].pct_change()
    
    # Align with market returns
    df = df.join(nifty_df['market_return'], how='left')
    
    # Rolling Beta (60-day window)
    covariance = df['return'].rolling(60).cov(df['market_return'])
    market_variance = df['market_return'].rolling(60).var()
    df['beta'] = covariance / market_variance
    df['beta'] = df['beta'].fillna(1.0) # Fallback to beta = 1
    
    # Residual return
    df['residual_return'] = df['return'] - df['beta'] * df['market_return']
    
    # Close Location Value (CLV)
    hl_range = df['high'] - df['low']
    df['clv'] = ((df['close'] - df['low']) - (df['high'] - df['close'])) / hl_range
    # Handle zero range (flat candles)
    df['clv'] = df['clv'].fillna(0.0)
    
    # Volume Indicators
    vol_mean = df['volume'].rolling(20).mean()
    vol_std = df['volume'].rolling(20).std().replace(0, 1e-6)
    df['vol_ratio'] = df['volume'] / vol_mean.replace(0, 1e-6)
    df['vol_z'] = (df['volume'] - vol_mean) / vol_std
    
    # Spread Indicators
    df['spread'] = df['high'] - df['low']
    spread_mean = df['spread'].rolling(20).mean()
    spread_std = df['spread'].rolling(20).std().replace(0, 1e-6)
    df['spread_z'] = (df['spread'] - spread_mean) / spread_std
    df['atr'] = df['spread'].rolling(20).mean()
    
    # Trend indicator (10-day EMA)
    df['ema10'] = df['close'].ewm(span=10, adjust=False).mean()
    
    # Save back
    dfs[symbol] = df
    print(f"Computed indicators for {symbol}. Latest data:\n{df.tail(3)}")

Computed indicators for RELIANCE. Latest data:
                             open    high     low   close    volume    return  \
date                                                                            
2026-08-26 00:00:00+05:30  1310.0  1315.6  1298.0  1298.0   5744474 -0.014427   
2026-08-27 00:00:00+05:30  1305.0  1308.4  1282.2  1282.2  11271497 -0.012173   
2026-08-28 00:00:00+05:30  1284.9  1291.8  1280.0  1287.0   6830228  0.003744   

                           market_return  beta  residual_return       clv  \
date                                                                        
2026-08-26 00:00:00+05:30      -0.005211   1.0        -0.009216 -1.000000   
2026-08-27 00:00:00+05:30      -0.004829   1.0        -0.007344 -1.000000   
2026-08-28 00:00:00+05:30       0.003520   1.0         0.000224  0.186441   

                           vol_ratio  vol_z  spread  spread_z  atr  \
date                                                                 
2026-08-26 00:00:00+0

## 5. Define Scoring Metrics

We implement three scores:

1. **Institutional Liquidity Absorption Score**:
   - Triggered on market down days (Nifty 50 return < -0.0075) where the stock return is $\ge 0$.
   - Formula: $Score = e_s \cdot VolRatio \cdot (1 + CLV)$

2. **Behavioral Exhaustion Score (Capitulation)**:
   - **Bearish Capitulation (Selling Climax)**:
     - Triggered when stock is below EMA10, Volume Z-score > 2.0, Spread Z-score > 1.5, and CLV > 0.4.
     - Formula: $Score = VolZ \cdot SpreadZ \cdot CLV$
   - **Bullish Capitulation (Blow-off Top)**:
     - Triggered when stock is above EMA10, Volume Z-score > 2.0, Spread Z-score > 1.5, and CLV < -0.4.
     - Formula: $Score = VolZ \cdot SpreadZ \cdot (-CLV)$

3. **Volatility Contraction Score (VCP)**:
   - Triggered when price is near 20-day high (within 4%), 5-day return volatility is less than 20-day volatility, and 5-day average volume is less than 20-day volume.
   - Formula: $Score = VolComp \cdot VoluCont \cdot (1 - DistFromHigh)$
     - $VolComp = \frac{\sigma_{Ret}(20) - \sigma_{Ret}(5)}{\sigma_{Ret}(20)}$
     - $VoluCont = 1 - \frac{MA(V, 5)}{MA(V, 20)}$
     - $DistFromHigh = \frac{\max(C_{t-20:t}) - C_t}{\max(C_{t-20:t})}$

In [ ]:
eval_date = pd.to_datetime('2026-08-28')
print(f"Evaluating scores for target date: {eval_date.date()}")

In [ ]:
absorption_results = []
bearish_exh_results = []
bullish_exh_results = []
vcp_results = []

nifty_row = nifty_df.loc[eval_date]
market_ret = nifty_row['market_return']
print(f"Nifty 50 return on eval date: {market_ret*100:.3f}%")

for symbol, df in dfs.items():
    if symbol == 'NIFTY 50' or 'NIFTY' in symbol:
        continue
        
    if eval_date not in df.index:
        continue
        
    row = df.loc[eval_date]
    prev_row = df.shift(1).loc[eval_date]
    
    # 1. Institutional Liquidity Absorption
    # Nifty drops significantly, stock is green/flat, showing relative strength and absorption
    # We trigger if Nifty is down < -0.5% (to make sure we get triggers if market was weak) and stock return >= 0
    if market_ret < -0.005 and row['return'] >= 0:
        abs_score = row['residual_return'] * row['vol_ratio'] * (1 + row['clv'])
        absorption_results.append({
            'Symbol': symbol,
            'Price': row['close'],
            'Stock Return (%)': round(row['return'] * 100, 2),
            'Residual Return (%)': round(row['residual_return'] * 100, 2),
            'Volume Ratio': round(row['vol_ratio'], 2),
            'CLV': round(row['clv'], 2),
            'Absorption Score': round(abs_score, 4)
        })
        
    # 2. Behavioral Exhaustion
    # Bearish Capitulation: downtrend, huge volume, wide spread, closing near high (reversal lower tail)
    # Bullish Capitulation: uptrend, huge volume, wide spread, closing near low (reversal upper tail)
    if row['vol_z'] > 1.5 and row['spread_z'] > 1.0:
        # Bearish Capitulation
        if row['close'] < row['ema10'] and row['clv'] > 0.3:
            bear_score = row['vol_z'] * row['spread_z'] * row['clv']
            bearish_exh_results.append({
                'Symbol': symbol,
                'Price': row['close'],
                'Volume Z-Score': round(row['vol_z'], 2),
                'Spread Z-Score': round(row['spread_z'], 2),
                'CLV': round(row['clv'], 2),
                'Bearish Exhaustion Score': round(bear_score, 2)
            })
        # Bullish Capitulation
        elif row['close'] > row['ema10'] and row['clv'] < -0.3:
            bull_score = row['vol_z'] * row['spread_z'] * (-row['clv'])
            bullish_exh_results.append({
                'Symbol': symbol,
                'Price': row['close'],
                'Volume Z-Score': round(row['vol_z'], 2),
                'Spread Z-Score': round(row['spread_z'], 2),
                'CLV': round(row['clv'], 2),
                'Bullish Exhaustion Score': round(bull_score, 2)
            })

    # 3. Volatility Contraction Pattern (VCP)
    # Near highs, vol and range shrinking
    # We look at historical 20-day window up to eval_date
    sub_df = df.loc[:eval_date].tail(20)
    if len(sub_df) == 20:
        high_20 = sub_df['close'].max()
        dist_from_high = (high_20 - row['close']) / high_20
        
        # Volatility of last 5 days vs 20 days
        vol_5d = sub_df['return'].tail(5).std()
        vol_20d = sub_df['return'].std()
        
        # Volume of last 5 days vs 20 days
        avg_vol_5d = sub_df['volume'].tail(5).mean()
        avg_vol_20d = sub_df['volume'].mean()
        
        if vol_20d > 0 and avg_vol_20d > 0:
            vol_comp = (vol_20d - vol_5d) / vol_20d
            volu_cont = 1.0 - (avg_vol_5d / avg_vol_20d)
            
            # Only trigger if there is actual contraction and we are near the high
            if vol_comp > 0 and volu_cont > 0 and dist_from_high < 0.05:
                vcp_score = vol_comp * volu_cont * (1.0 - dist_from_high)
                vcp_results.append({
                    'Symbol': symbol,
                    'Price': row['close'],
                    'Dist from High (%)': round(dist_from_high * 100, 2),
                    'Vol Comp (%)': round(vol_comp * 100, 2),
                    'Volume Contraction (%)': round(volu_cont * 100, 2),
                    'VCP Score': round(vcp_score, 4)
                })


## 6. Display Rankings

We sort and display the top stocks for each outcome.

In [ ]:
abs_df = pd.DataFrame(absorption_results).sort_values(by='Absorption Score', ascending=False)
bear_df = pd.DataFrame(bearish_exh_results).sort_values(by='Bearish Exhaustion Score', ascending=False)
bull_df = pd.DataFrame(bullish_exh_results).sort_values(by='Bullish Exhaustion Score', ascending=False)
vcp_df = pd.DataFrame(vcp_results).sort_values(by='VCP Score', ascending=False)

print("=== TOP 5 STOCKS: INSTITUTIONAL LIQUIDITY ABSORPTION ===")
print(abs_df.head(5).to_string(index=False))

print("\n=== TOP 5 STOCKS: BEARISH CAPITULATION (SELLING EXHAUSTION) ===")
print(bear_df.head(5).to_string(index=False))

print("\n=== TOP 5 STOCKS: BULLISH CAPITULATION (BUYING EXHAUSTION) ===")
print(bull_df.head(5).to_string(index=False))

print("\n=== TOP 5 STOCKS: VOLATILITY CONTRACTION (VCP) ===")
print(vcp_df.head(5).to_string(index=False))